<a href="https://colab.research.google.com/github/root-epifit/HPC_Sber_2025/blob/main/Lec9_SberUniversitet_MPI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mpi4py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 15.0 MB/s eta 0:00:00


In [ ]:
from mpi4py import MPI

comm = MPI.COMM_WORLD
my_rank = comm.Get_rank()
mpi_size = comm.Get_size()
print(mpi_size, my_rank)



1 0


In [ ]:
%%writefile test_mpi.py
from mpi4py import MPI
from time import sleep

comm = MPI.COMM_WORLD
my_rank = comm.Get_rank()
size = comm.Get_size()

print(f'Hello, my rank is {my_rank}, total size is {size}')


if my_rank==0:
    sleep(3)
elif my_rank==1:
    sleep(4)
elif my_rank==2:
    sleep(1)

# image_000.jpg
# image_001.jpg
# ...
# image_999.jpg

# filename = image + rank . jpg

# python code


MPI.Finalize()

Writing test_mpi.py


In [ ]:
!python test_mpi.py

Hello, my rank is 0, total size is 1


In [ ]:
!time mpirun --allow-run-as-root --oversubscribe -n 3 python test_mpi.py

Hello, my rank is 0, total size is 3
Hello, my rank is 1, total size is 3
Hello, my rank is 2, total size is 3

real	0m4.643s
user	0m0.507s
sys	0m0.367s


In [ ]:
%%writefile mpi_send.py

from mpi4py import MPI

comm = MPI.COMM_WORLD
my_rank = comm.Get_rank()

if my_rank == 0:
    a = ['fdasdfa', 1231213, 1, 3.3, {'ddd': 15}, (1,3)]
    comm.send(a, dest=1, tag=0)
    # send is non-blocking
    # rabota
    # rabota
elif my_rank == 1:
    a = comm.recv(source=0, tag=0)
    print(a)

MPI.Finalize()

Overwriting mpi_send.py


In [ ]:
%%writefile mpi_send_numpy_1.py

from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
my_rank = comm.Get_rank()

if my_rank == 0:
    a = np.linspace(0,10,10,endpoint=False, dtype=np.int32)
    comm.send(a, dest=1, tag=0)
    # send is non-blocking
    # rabota
    # rabota
elif my_rank == 1:
    a = comm.recv(source=0, tag=0)
    print(a)

MPI.Finalize()

Overwriting mpi_send_numpy_1.py


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -n 2 python mpi_send.py

In [ ]:
!mpirun --allow-run-as-root --oversubscribe -n 2 python mpi_send_numpy_1.py

[0 1 2 3 4 5 6 7 8 9]


In [ ]:
%%writefile mpi_send.py
from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
my_rank = comm.Get_rank()

n_of_elements = 2**22

if my_rank == 0:
    a = np.linspace(0, n_of_elements, n_of_elements, endpoint=False, dtype=np.uint32)
    comm.send(a, dest=1, tag=0)

elif my_rank == 1:
    #a = np.zeros(n_of_elements, dtype=np.uint32)
    a=comm.recv(source=0, tag=0)

    print(a)

MPI.Finalize()

In [ ]:
!mpirun --allow-run-as-root --oversubscribe -n 2 python mpi_send.py

In [ ]:
%%writefile mpi_send_numpy.py

from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
my_rank = comm.Get_rank()

n_of_elements = 16

if my_rank == 0:
    a = np.linspace(0, n_of_elements, n_of_elements, endpoint=False, dtype=np.uint32)
    comm.Send([a, 8, MPI.UNSIGNED], dest=1, tag=0)

elif my_rank == 1:
    a = np.zeros(n_of_elements, dtype=np.uint32)
    comm.Recv([a, 8, MPI.UNSIGNED], source=0, tag=0)

    print(a)

MPI.Finalize()

Overwriting mpi_send_numpy.py


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -n 2 python mpi_send_numpy.py

[0 1 2 3 4 5 6 7 0 0 0 0 0 0 0 0]


In [ ]:
%%writefile mpi_reduce.py

from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
my_rank = comm.Get_rank()

a = np.array([my_rank, my_rank+1], dtype=np.uint8)

# my_rank = 0 --> [0, 1]
# my_rank = 1 --> [1, 2]

print(a)

root_process = 0

result = None
if my_rank == root_process:
    result = np.empty(2, dtype=np.uint8)

comm.Reduce([a, 2, MPI.UNSIGNED_CHAR], [result, 2, MPI.UNSIGNED_CHAR], op=MPI.SUM, root=root_process)

if my_rank == root_process:
    print("Final result = ", result)

MPI.Finalize()

Overwriting mpi_reduce.py


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -n 4 python mpi_reduce.py

[2 3]
[3 4]
[0 1]
[1 2]
Final result =  [ 6 10]


In [ ]:
%%writefile mpi_allreduce.py

from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
my_rank = comm.Get_rank()

a = np.array([my_rank, my_rank+1], dtype=np.int32)

result = np.empty(2, dtype=np.int32)

comm.Allreduce([a, 2, MPI.INT], [result, 2, MPI.INT], op=MPI.SUM)

print(result)

Writing mpi_allreduce.py


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -n 4 python mpi_allreduce.py

[ 6 10]
[ 6 10]
[ 6 10]
[ 6 10]


This is a copy-paste from ChatGPT 4o:

# Spark vs MPI in Python: Use Case Comparison and Time-to-Market Analysis

This document provides a practical and precise comparison between **Apache Spark** (via PySpark) and **MPI** (via `mpi4py`) for distributed computing tasks in Python. It focuses on **programming paradigms**, **performance**, **time-to-market**, and **real-world use cases**.

---

## 1. Conceptual Comparison

| Aspect | Spark (PySpark) | MPI (mpi4py) |
|--------|----------------|--------------|
| **Paradigm** | Data-parallel, functional (map/reduce) | Message-passing parallelism |
| **Programming Model** | High-level, declarative | Low-level, imperative |
| **Fault Tolerance** | Built-in (RDD lineage, retries) | None (crashes must be handled manually) |
| **Target Environment** | Commodity clusters, cloud, HDFS/S3 | HPC clusters, low-latency interconnect |
| **Ease of Use** | Easier to learn and prototype | Steeper learning curve |
| **Best For** | Big data analytics, ETL, ML pipelines | Tightly coupled simulations, numerical solvers |

---

## 2. Time-to-Market Comparison

- **Spark**
  - ✅ Fast prototyping, high-level APIs
  - ✅ Easy integration with data sources (Parquet, CSV, S3, databases)
  - ❌ Overhead due to JVM/Python bridge (PySpark)
  - ✅ Popular with data engineers and ML engineers

- **MPI**
  - ❌ Requires knowledge of parallel algorithms and data layout
  - ❌ Manual communication and synchronization
  - ✅ Near-native HPC performance
  - ✅ Ideal for performance-critical simulations

> 💡 **Verdict**: Spark wins for data-centric tasks with fast deployment needs. MPI wins for raw speed in simulations.

---

## 3. Use Case Comparison

### Spark (PySpark) is Best For:
- Distributed **ETL pipelines**
- Processing large CSV/Parquet/JSON datasets
- Distributed machine learning preprocessing
- Log processing and aggregations
- Ad-hoc or scheduled batch analytics

### MPI (mpi4py) is Best For:
- Large-scale **scientific simulations**
- Solving PDEs (finite difference, finite element methods)
- Distributed linear algebra (matrix ops, eigensolvers)
- Custom communication patterns (e.g., nearest-neighbor exchange)
- High-performance Monte Carlo methods

---

## 4. Code Examples

### Spark Example (PySpark)
```python
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("LogAggregator").getOrCreate()

df = spark.read.json("s3://mybucket/logs/")
result = df.groupBy("user_id").count()
result.write.parquet("s3://mybucket/output/")
```

```python
from mpi4py import MPI

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

data = rank ** 2
sum_data = comm.reduce(data, op=MPI.SUM, root=0)

if rank == 0:
    print("Sum:", sum_data)

```

## 4. When to Use Which?

| Scenario                                        | Use Spark | Use MPI |
|------------------------------------------------|-----------|---------|
| Log aggregation over 500 GB                    | ✅        | ❌      |
| CFD simulation (fluid dynamics)                | ❌        | ✅      |
| ML preprocessing (ETL + feature engineering)   | ✅        | ❌      |
| Nonlinear PDE solver with neighbor exchange    | ❌        | ✅      |
| Business dashboards (SQL + Parquet)            | ✅        | ❌      |
| Particle simulation or ray tracing             | ❌        | ✅      |
| Interactive data exploration                   | ✅        | ❌      |
| Fast, low-level control of parallelism         | ❌        | ✅      |
| Fault-tolerant processing on commodity clusters| ✅        | ❌      |

✅ = Recommended  
❌ = Not suitable / inefficient

---

## 5. Hybrid Pattern

Many production systems **combine Spark and MPI**:

> ⚙️ Spark handles **data ingestion**, **ETL**, and **filtering**  
> 🚀 MPI handles **tight-loop physics**, **simulation**, or **scientific compute**

**Example Pipeline:**
1. Use Spark to filter and preprocess 100 TB of raw experimental data.
2. Save results to HDF5 or Parquet.
3. Launch MPI code to simulate physical models (e.g., ray tracing or N-body simulation).

This hybrid pattern gives **developer productivity** on the front end and **performance/scalability** on the backend.


---

## Summary

| Trait                         | Spark                   | MPI                        |
|------------------------------|--------------------------|-----------------------------|
| High-Level APIs              | ✅                        | ❌ (low-level primitives)   |
| Performance (tight loops)    | ❌                        | ✅                          |
| Tolerance to node failure    | ✅ (lineage-based retry)  | ❌                          |
| Learning Curve               | Low                      | High                        |
| Ideal for ETL/analytics      | ✅                        | ❌                          |
| Ideal for numerical solvers  | ❌                        | ✅                          |
| Ecosystem                    | Strong (MLlib, Delta)    | Niche (scientific only)     |

- Use **Spark** for scalable **data processing pipelines**.
- Use **MPI** for **scientific simulations** and **performance-critical code**.
- Combine both for **maximum flexibility and power**.

---


In [ ]:
dN = 22
procs = 4

my_N = [0, 0, 0, 0]

for i in range(procs):
    my_N[i] = N // procs
    if i < N % procs:
        my_N[i]+=1

print(my_N)

In [ ]:
2**21 / 4

524288.0

In [ ]:
%%writefile mpi_pi.py

from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
my_rank = comm.Get_rank()
n_processes = comm.Get_size()

# 16 tasks 3 process:     0: 6  1: 5  2: 5
#

global_N = 2**21

# 2**21 / 4 = 2**19

my_N = global_N // n_processes
if (my_rank < global_N % n_processes):
    my_N += 1

start_time = MPI.Wtime()

my_counter = 0

for _ in range(my_N):
    x,y = np.random.rand(2)
    if (x**2+y**2)<1:
        my_counter += 1

my_pi = my_counter / my_N * 4

root = 0

if my_rank == root:
    global_pi = 0.0
else:
    global_pi = None

global_pi = comm.reduce(my_pi, op=MPI.SUM, root=root)

if my_rank == root:
    global_pi = global_pi / n_processes

end_time = MPI.Wtime()
tot_time = end_time - start_time

if my_rank==0:
    print(f"I am {my_rank}, {my_pi=}, {global_pi=}, {tot_time=}")

Overwriting mpi_pi.py


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -n 2 python mpi_pi.py

I am 0, my_pi=3.1400680541992188, global_pi=3.1415557861328125, tot_time=8.097249801


In [ ]:
%%writefile bcast.py
#### Пример использования broadcast для разливания параметров

from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

# On rank 0, initialize simulation parameters
if rank == 0:
    params = {
        "dt": 0.01,                     # time step
        "num_steps": 100,               # number of iterations
        "domain_size": (100, 100),      # grid dimensions
        "initial_temperature": 300.0,   # Kelvin
        "diffusion_coeff": 0.1          # arbitrary units
    }
    print(f"[Rank {rank}] Initialized parameters: {params}")
else:
    params = None  # Placeholder for receiving

# Broadcast parameters from rank 0 to all ranks
params = comm.bcast(params, root=0)

# Now every rank can use `params` identically
print(f"[Rank {rank}] Received parameters: {params}")

# Example: distribute grid initialization
# Each process will get part of the domain along the first axis
rows_per_rank = params["domain_size"][0] // size
start_row = rank * rows_per_rank
end_row = start_row + rows_per_rank

local_grid = np.full(
    (rows_per_rank, params["domain_size"][1]),
    params["initial_temperature"],
    dtype=np.float64
)

print(f"[Rank {rank}] Local grid shape: {local_grid.shape}, "
      f"rows: {start_row}-{end_row-1}")


Writing bcast.py


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -n 2 python bcast.py

[Rank 0] Initialized parameters: {'dt': 0.01, 'num_steps': 100, 'domain_size': (100, 100), 'initial_temperature': 300.0, 'diffusion_coeff': 0.1}
[Rank 0] Received parameters: {'dt': 0.01, 'num_steps': 100, 'domain_size': (100, 100), 'initial_temperature': 300.0, 'diffusion_coeff': 0.1}
[Rank 0] Local grid shape: (50, 100), rows: 0-49
[Rank 1] Received parameters: {'dt': 0.01, 'num_steps': 100, 'domain_size': (100, 100), 'initial_temperature': 300.0, 'diffusion_coeff': 0.1}
[Rank 1] Local grid shape: (50, 100), rows: 50-99


Асинхронная пересылка
=====

In [ ]:
%%writefile isend.py
from mpi4py import MPI
import numpy as np
import time

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

# Simple 1D ring topology
next_rank = (rank + 1) % size
prev_rank = (rank - 1) % size

# Data to send (simulate "boundary" values)
send_data = np.full(1000000, rank, dtype=np.float64)  # 1 million doubles
recv_data = np.empty_like(send_data)

# --- Start non-blocking communication ---
req_send = comm.Isend(send_data, dest=next_rank, tag=0)
req_recv = comm.Irecv(recv_data, source=prev_rank, tag=0)

# --- Overlap with computation ---
# Here we simulate expensive work while data transfers happen
t0 = time.time()
local_sum = 0
for i in range(5000000):  # Some dummy computation
    local_sum += i % 7
t1 = time.time()

print(f"[Rank {rank}] Finished local computation in {t1 - t0:.3f} s "
      f"while communicating.")

# --- Wait for communication to finish ---
req_send.Wait()
req_recv.Wait()

# Now we can use received data
print(f"[Rank {rank}] Received sum: {np.sum(recv_data)} "
      f"from rank {prev_rank}")


Writing isend.py


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -n 2 python isend.py

[Rank 1] Finished local computation in 1.320 s while communicating.
[Rank 0] Finished local computation in 1.453 s while communicating.
[Rank 0] Received sum: 1000000.0 from rank 1
[Rank 1] Received sum: 0.0 from rank 0


Кастомные типы данных
======

In [ ]:
%%writefile custom_dtype.py
from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

# --- Define a structured dtype ---
particle_dtype = np.dtype([
    ('position', np.float64, 3),  # 3D position (x, y, z)
    ('velocity', np.float64, 3)   # 3D velocity (vx, vy, vz)
])

# AoS vs SoA


# --- Create an array of particles on rank 0 ---
if rank == 0:
    num_particles = 5
    particles = np.zeros(num_particles, dtype=particle_dtype)

    # Fill with some example data
    particles['position'] = np.random.rand(num_particles, 3) * 10.0
    particles['velocity'] = np.random.randn(num_particles, 3)
    print(f"[Rank 0] Initial particles:\n{particles}\n")
else:
    particles = None

# --- Broadcast number of particles first ---
num_particles = comm.bcast(len(particles) if rank == 0 else None, root=0)

# --- Allocate receive array for all other ranks ---
if rank != 0:
    particles = np.empty(num_particles, dtype=particle_dtype)

# --- Broadcast the structured array ---
comm.Bcast([particles, MPI.DOUBLE], root=0)

print(f"[Rank {rank}] Received particles:\n{particles}\n")


Writing custom_dtype.py


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -n 2 python custom_dtype.py

[Rank 0] Initial particles:
[([1.58597252, 3.65427462, 4.12397401], [ 0.51953915,  0.08495414,  0.49667633])
 ([3.36070316, 9.35516274, 4.26147325], [ 1.98414279, -1.46579469,  0.92689399])
 ([6.37984318, 1.00062621, 5.39129348], [-1.44349027,  0.04245563,  0.94310697])
 ([5.66325208, 3.14215811, 3.54297873], [ 0.52614978, -1.13124789, -1.41396664])
 ([1.10219213, 9.16440426, 7.11785254], [-0.24306467,  0.28859811, -2.57595447])]

[Rank 0] Received particles:
[([1.58597252, 3.65427462, 4.12397401], [ 0.51953915,  0.08495414,  0.49667633])
 ([3.36070316, 9.35516274, 4.26147325], [ 1.98414279, -1.46579469,  0.92689399])
 ([6.37984318, 1.00062621, 5.39129348], [-1.44349027,  0.04245563,  0.94310697])
 ([5.66325208, 3.14215811, 3.54297873], [ 0.52614978, -1.13124789, -1.41396664])
 ([1.10219213, 9.16440426, 7.11785254], [-0.24306467,  0.28859811, -2.57595447])]

[Rank 1] Received particles:
[([1.58597252, 3.65427462, 4.12397401], [ 0.51953915,  0.08495414,  0.49667633])
 ([3.36070316, 9.35

In [ ]:
%%writefile create_vector.py

from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

if size < 2:
    if rank == 0:
        print("Run with at least 2 ranks.")
    raise SystemExit

# Square matrix size
N = 6
i_row = 2
j_col = 4

# Create matrix in C-order layout
if rank == 0:
    A = np.arange(N * N, dtype=np.float64).reshape(N, N, order='C')
    print("[Rank 0] Matrix A:\n", A)
else:
    A = np.empty((N, N), dtype=np.float64, order='C')  # not used, but keeps code symmetric

# ---- Create derived datatypes ----

itemsize = np.dtype(np.float64).itemsize

# 1. Row: N elements, stride 1 → contiguous row
rowtype = MPI.DOUBLE.Create_vector(count=N, blocklength=1, stride=1)
rowtype = rowtype.Create_resized(0, itemsize)
rowtype.Commit()

# 2. Column: N elements, stride N (one element per row)
coltype = MPI.DOUBLE.Create_vector(count=N, blocklength=1, stride=N)
coltype = coltype.Create_resized(0, itemsize)
coltype.Commit()

# 3. Diagonal: stride N+1
diagtype = MPI.DOUBLE.Create_vector(count=N, blocklength=1, stride=N + 1)
diagtype = diagtype.Create_resized(0, itemsize)
diagtype.Commit()

# ---- Allocate receive buffers ----
if rank == 1:
    recv_row = np.empty(N, dtype=np.float64)
    recv_col = np.empty(N, dtype=np.float64)
    recv_diag = np.empty(N, dtype=np.float64)

# ---- Send / Receive operations ----
if rank == 0:
    # Send row i_row: use full A and derived type, starting at A[i_row, 0]
    comm.Send([A[i_row, 0:], 1, rowtype], dest=1, tag=10)

    # Send column j_col: use A[0, j_col:] to ensure base is in correct location
    comm.Send([A[0, j_col:], 1, coltype], dest=1, tag=20)

    # Send diagonal: can just send from full A
    comm.Send([A, 1, diagtype], dest=1, tag=30)

elif rank == 1:
    comm.Recv([recv_row, MPI.DOUBLE], source=0, tag=10)
    print(f"[Rank 1] Received row {i_row}:", recv_row)

    comm.Recv([recv_col, MPI.DOUBLE], source=0, tag=20)
    print(f"[Rank 1] Received column {j_col}:", recv_col)

    comm.Recv([recv_diag, MPI.DOUBLE], source=0, tag=30)
    print(f"[Rank 1] Received main diagonal:", recv_diag)

# ---- Clean up ----
rowtype.Free()
coltype.Free()
diagtype.Free()


MPI.Finalize()

Writing create_vector.py


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -n 2 python create_vector.py

[Rank 0] Matrix A:
 [[ 0.  1.  2.  3.  4.  5.]
 [ 6.  7.  8.  9. 10. 11.]
 [12. 13. 14. 15. 16. 17.]
 [18. 19. 20. 21. 22. 23.]
 [24. 25. 26. 27. 28. 29.]
 [30. 31. 32. 33. 34. 35.]]
[Rank 1] Received row 2: [12. 13. 14. 15. 16. 17.]
[Rank 1] Received column 4: [ 4. 10. 16. 22. 28. 34.]
[Rank 1] Received main diagonal: [ 0.  7. 14. 21. 28. 35.]




Измерение времени
----

In [ ]:
%%writefile use_wtime.py
### From https://education.molssi.org/parallel-programming/03-distributed-examples-mpi4py.html

import numpy as np
from mpi4py import MPI

if __name__ == "__main__":

    # get basic information about the MPI communicator
    world_comm = MPI.COMM_WORLD
    world_size = world_comm.Get_size()
    my_rank = world_comm.Get_rank()

    N = 10000000

    # initialize a
    start_time = MPI.Wtime()
    a = np.ones( N )
    end_time = MPI.Wtime()
    if my_rank == 0:
        print("Initialize a time: " + str(end_time-start_time))

    # initialize b
    start_time = MPI.Wtime()
    b = np.zeros( N )
    for i in range( N ):
        b[i] = 1.0 + i
    end_time = MPI.Wtime()
    if my_rank == 0:
        print("Initialize b time: " + str(end_time-start_time))

    # add the two arrays
    start_time = MPI.Wtime()
    for i in range( N ):
        a[i] = a[i] + b[i]
    end_time = MPI.Wtime()
    if my_rank == 0:
        print("Add arrays time: " + str(end_time-start_time))

    # average the result
    start_time = MPI.Wtime()
    sum = 0.0
    for i in range( N ):
        sum += a[i]
    average = sum / N
    end_time = MPI.Wtime()
    if my_rank == 0:
        print("Average result time: " + str(end_time-start_time))
        print("Average: " + str(average))

Writing use_wtime.py


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -n 4 python use_wtime.py

Initialize a time: 0.117505561
Initialize b time: 10.350810675
Add arrays time: 23.787250157000003
Average result time: 11.955088242999999
Average: 5000001.5


Параллельный ввод/вывод
------

In [ ]:
%%writefile test_io.py

from mpi4py import MPI
import numpy as np

amode = MPI.MODE_WRONLY|MPI.MODE_CREATE
comm = MPI.COMM_WORLD
fh = MPI.File.Open(comm, "datafile.config", amode)

buffer = np.empty(10, dtype=np.int32)
buffer[:] = comm.Get_rank()

## transfer buffer to GPU
## postprocess on GPU
## transfer to CPU
## use MPI for distributing between servers

print(buffer)

offset = comm.Get_rank()*buffer.nbytes
fh.Write_at_all(offset, buffer)


fh.Close()
MPI.Finalize()

Writing test_io.py


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -n 16 python test_io.py

[8 8 8 8 8 8 8 8 8 8]
[0 0 0 0 0 0 0 0 0 0]
[6 6 6 6 6 6 6 6 6 6]
[12 12 12 12 12 12 12 12 12 12]
[7 7 7 7 7 7 7 7 7 7]
[15 15 15 15 15 15 15 15 15 15]
[3 3 3 3 3 3 3 3 3 3]
[4 4 4 4 4 4 4 4 4 4]
[9 9 9 9 9 9 9 9 9 9]
[13 13 13 13 13 13 13 13 13 13]
[14 14 14 14 14 14 14 14 14 14]
[5 5 5 5 5 5 5 5 5 5]
[10 10 10 10 10 10 10 10 10 10]
[11 11 11 11 11 11 11 11 11 11]
[1 1 1 1 1 1 1 1 1 1]
[2 2 2 2 2 2 2 2 2 2]


In [ ]:
!hexdump -C datafile.config

In [ ]:
import numpy as np

data = np.fromfile("datafile.config", dtype=np.int32)

print(data)

[ 0  0  0  0  0  0  0  0  0  0  1  1  1  1  1  1  1  1  1  1  2  2  2  2
  2  2  2  2  2  2  3  3  3  3  3  3  3  3  3  3  4  4  4  4  4  4  4  4
  4  4  5  5  5  5  5  5  5  5  5  5  6  6  6  6  6  6  6  6  6  6  7  7
  7  7  7  7  7  7  7  7  8  8  8  8  8  8  8  8  8  8  9  9  9  9  9  9
  9  9  9  9 10 10 10 10 10 10 10 10 10 10 11 11 11 11 11 11 11 11 11 11
 12 12 12 12 12 12 12 12 12 12 13 13 13 13 13 13 13 13 13 13 14 14 14 14
 14 14 14 14 14 14 15 15 15 15 15 15 15 15 15 15]


Виртуальная топология
------

In [ ]:
%%writefile test_topology.py
from mpi4py import MPI

# This is to create default communicator and get the rank
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
cartesian3d = comm.Create_cart(dims = [3,3,3],periods =[True,True,True],reorder=False)
coord3d = cartesian3d.Get_coords(rank)
print ("In 3D topology, Processor ",rank, " has coordinates ",coord3d)

comm.Barrier()
# Get coordinates of your neighbour to left and right
left,right = cartesian3d.Shift(direction = 0,disp=2)
up,down = cartesian3d.Shift(direction = 1,disp=1)
north,south = cartesian3d.Shift(direction = 2,disp=1)

# comm.send(a, dest=right, tag=0)
# b=comm.recv(source=left, tag=0)

#up,down = cartesian3d.Shift(direction = 1,disp=2)

print("Processor ",rank, "has his neighbour in 0-direction", left, " and ",right)

Overwriting test_topology.py


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -n 27 python test_topology.py

In 3D topology, Processor  0  has coordinates  [0, 0, 0]
In 3D topology, Processor  1  has coordinates  [0, 0, 1]
In 3D topology, Processor  17  has coordinates  [1, 2, 2]
In 3D topology, Processor  4  has coordinates  [0, 1, 1]
In 3D topology, Processor  20  has coordinates  [2, 0, 2]
In 3D topology, Processor  3  has coordinates  [0, 1, 0]
In 3D topology, Processor  11  has coordinates  [1, 0, 2]
In 3D topology, Processor  8  has coordinates  [0, 2, 2]
In 3D topology, Processor  25  has coordinates  [2, 2, 1]
In 3D topology, Processor  16  has coordinates  [1, 2, 1]
In 3D topology, Processor  2  has coordinates  [0, 0, 2]
In 3D topology, Processor  5  has coordinates  [0, 1, 2]
In 3D topology, Processor  19  has coordinates  [2, 0, 1]
In 3D topology, Processor  15  has coordinates  [1, 2, 0]
In 3D topology, Processor  10  has coordinates  [1, 0, 1]
In 3D topology, Processor  23  has coordinates  [2, 1, 2]
In 3D topology, Processor  13  has coordinates  [1, 1, 1]
In 3D topology, Proce

In [ ]:
%%writefile openmp.c

#include <iostream>
#include <omp.h>
#include <array>


#define N 1<<22

int main() {

  std::cout << "Hello world\n";
  double a[N] = {1};
  double b[N] = {2};
  double c[N] = {0};

// init



#pragma omp parallel for
  for (auto n = 0; n < N; ++n) {
      a[n] = 1;
      b[n] = 2;
  }

//////

#pragma omp parallel for
  for (auto n = 0; n < N; ++n) {
      c[n] = a[n] + b[n];
  }



///// printer
//  for (auto n = 0; n < 10; ++n) {
//      std::cout << c[n] << std::endl;
//  }

  return 0;
}

Overwriting openmp.c


In [ ]:
!g++ openmp.c -fopenmp -o test

In [ ]:
!time ./test

Hello world

real	0m0.113s
user	0m0.065s
sys	0m0.070s


In [ ]:
# 1 worker per node (process per node)   ---> MPI
# 1 worker spawns 32 thread              ---> OpenMP
# 1 worker loads GPU                     ---> Cuda